# D2.6 · Post-incident change surface

**Function D — AI for SecOps → The Incident Responder**  ·  *Security of AI*

Builds on **[D2.5 · Replay and forensics](https://spbreed.github.io/cyber-commons/lessons/D2.5.html)**.

| | |
|---|---|
| Open-source tooling | — |
| Open-weight models | — |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The hook

After an agentic incident the change surface is not the code. It is prompts, tool scopes, model versions and policy — four things with no release process, no review and, usually, no version history.

## 2 · The framework

```
   the change surface after an agentic incident

   +---------+ +--------+ +----------+ +---------+
   | prompts | | scopes | | model ver| | policy  |
   +---------+ +--------+ +----------+ +---------+
        no release process . no review . no history

   a fix in any of the four is invisible unless it is versioned
```

After an incident you change something. For ordinary software that change goes
through code review, CI and a deploy — a process that records what changed and
who approved it.

For an agent the fix may be a prompt, a tool manifest, a model version, a policy
file or an approval toggle. Only some of those go through any process at all,
and the ones that do not are precisely the ones most likely to be adjusted at
2am during an incident.

The consequence is a system whose security-relevant configuration drifts with no
record, and a post-incident action list where half the items cannot be verified
as done six weeks later.

## 3 · Demo — where post-incident changes actually land

In [ ]:
SURFACES = {
 "application code":   ("yes",       "PR, review, CI, deploy"),
 "agent prompt":       ("no",        "edited in a console, no diff retained"),
 "tool manifest":      ("no",        "config change; no threat-model diff (A1.1)"),
 "model version":      ("no",        "provider-side; you may not be told"),
 "policy (in git)":    ("yes",       "if it is in git — often it is not"),
 "approval settings":  ("no",        "a toggle in an admin UI"),
 "egress allowlist":   ("sometimes", "depends whether it is IaC or a console"),
}
print(f"{'change surface':22s}{'in change mgmt?':18s}what happens today")
print("-" * 76)
for k, (managed, how) in SURFACES.items():
    print(f"{k:22s}{managed:18s}{how}")
unmanaged = [k for k, (m, _) in SURFACES.items() if m == "no"]
print(f"\n{len(unmanaged)}/{len(SURFACES)} bypass change management: {unmanaged}")

## 4 · Where it breaks — six weeks later

In [ ]:
ACTIONS = [
 ("revoke the compromised agent identity", "identity provider", True),
 ("add collect.example.com to the egress denylist", "console", False),
 ("remove read access to /home/app/.aws", "tool manifest", False),
 ("require approval for http_post", "admin toggle", False),
 ("add a regression test for the credential read", "code", True),
 ("update the prompt to warn about credential files", "console", False),
]
print(f"{'action':48s}{'landed in':20s}verifiable in 6 weeks?")
print("-" * 92)
for a, where, verifiable in ACTIONS:
    print(f"{a:48s}{where:20s}{verifiable}")
v = sum(x[2] for x in ACTIONS)
print(f"\n{v}/{len(ACTIONS)} post-incident actions can be verified later.")
print("The other four exist only in the incident document.")

print("\nAlso note action 6: 'update the prompt to warn about credential files'.")
print("That is a request for the model to behave better. It is not a control,")
print("and it will be silently reverted by the next prompt edit.")

## 5 · The control — the manifest diff, and a verification date

In [ ]:
SCOPE_WEIGHT = {"self": 1, "project": 3, "tenant": 8, "org": 20}
def blast(tools, gated=frozenset()):
    return sum(SCOPE_WEIGHT[s]*(1 if rev else 2) for n, s, rev in tools if n not in gated)

BEFORE = [("read_file", "self", True), ("write_file", "project", True),
          ("http_post", "org", False)]
AFTER  = [("read_file", "self", True), ("write_file", "project", True),
          ("http_post", "org", False)]
gated_after = {"http_post"}

print(f"blast before {blast(BEFORE)}  after {blast(AFTER, gated_after)}")
print("the manifest diff records the change even though no PR was raised.\n")

def action_record(action, surface, control_type, owner, verify_by):
    is_control = control_type in ("preventive", "detective")
    return {"action": action, "surface": surface, "type": control_type,
            "owner": owner, "verify_by": verify_by,
            "acceptable": is_control and bool(owner) and bool(verify_by)}

RECORDS = [
 action_record("gate http_post behind approval", "tool manifest", "preventive",
               "platform-sec", "2026-09-30"),
 action_record("alert on credential-path reads", "detection", "detective",
               "soc", "2026-09-15"),
 action_record("update the prompt to warn the model", "prompt", "guidance",
               "", ""),
]
for r in RECORDS:
    print(f"{'OK  ' if r['acceptable'] else 'WEAK'} {r['action']:42s}"
          f"type={r['type']:11s} owner={r['owner'] or '—':14s} verify_by={r['verify_by'] or '—'}")
weak = [r for r in RECORDS if not r["acceptable"]]
print(f"\n{len(weak)} action(s) are guidance rather than controls: "
      f"{[r['action'] for r in weak]}")
assert weak

## What you just proved

Four of seven change surfaces bypass change management. Only 2 of 6 post-incident actions are verifiable six weeks later, and one of them is a prompt edit that is guidance rather than a control. The manifest diff records the gating change with the blast radius dropping from 40 to 3, and the action review flags the prompt update as weak.

## Your turn

Take your last incident's action list and mark each item's landing surface. Anything landing in a console has no record and no verification path — move those into git before the next one.

---

**Next → [D2.7 · Stop authority](https://spbreed.github.io/cyber-commons/lessons/D2.7.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/D2.6.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/D2.6.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*